# LangGraph 008 — Conditional Workflows

One routing function, one `add_conditional_edges` call, and only the branch the
input calls for. Needs `langgraph`; **no API key** — the classifier is scripted.

| Part | What we check |
|---|---|
| A | the quadratic solver across all three signs of `d` |
| B | 5 nodes defined, **3** ran — read from the checkpoint history |
| C | what a router may return, including the **silent typo** |
| D | a router that returns a list **fans out** |
| E | support routing: two branches of different lengths |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — A decision with three outcomes

In [ ]:
import math
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

class QuadState(TypedDict):
    a: int
    b: int
    c: int
    equation: str
    discriminant: float
    result: str

def show_equation(state: QuadState) -> dict:
    return {"equation": f"{state['a']}x2 {state['b']:+}x {state['c']:+}"}

def calculate_discriminant(state: QuadState) -> dict:
    return {"discriminant": state["b"] ** 2 - 4 * state["a"] * state["c"]}

def real_roots(state: QuadState) -> dict:
    root = math.sqrt(state["discriminant"])
    r1 = (-state["b"] + root) / (2 * state["a"])
    r2 = (-state["b"] - root) / (2 * state["a"])
    return {"result": f"Two real roots: {r1:.2f} and {r2:.2f}"}

def repeated_roots(state: QuadState) -> dict:
    return {"result": f"One repeated root: {-state['b'] / (2 * state['a']):.2f}"}

def no_real_roots(state: QuadState) -> dict:
    return {"result": "No real roots"}

# the routing function. It returns a NODE NAME, and changes nothing in the state.
def check_condition(state: QuadState) -> Literal["real_roots", "repeated_roots",
                                                 "no_real_roots"]:
    if state["discriminant"] > 0:
        return "real_roots"
    if state["discriminant"] == 0:
        return "repeated_roots"
    return "no_real_roots"

graph = StateGraph(QuadState)
graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_roots", real_roots)
graph.add_node("repeated_roots", repeated_roots)
graph.add_node("no_real_roots", no_real_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant", check_condition)
for node in ("real_roots", "repeated_roots", "no_real_roots"):
    graph.add_edge(node, END)

app = graph.compile()
for a, b, c in ((4, -5, -4), (1, -4, 4), (2, 4, 4)):
    out = app.invoke({"a": a, "b": b, "c": c,
                      "equation": "", "discriminant": 0.0, "result": ""})
    print(out["equation"], "| d =", out["discriminant"], "|", out["result"])

# 4x2 -5x -4 | d = 89  | Two real roots: 1.80 and -0.55
# 1x2 -4x +4 | d = 0   | One repeated root: 2.00
# 2x2 +4x +4 | d = -16 | No real roots

In [ ]:
cases = {(4, -5, -4): 89, (1, -4, 4): 0, (2, 4, 4): -16, (1, 0, -9): 36, (3, 6, 3): 0}
for (a, b, c), expected_d in cases.items():
    out = app.invoke({"a": a, "b": b, "c": c,
                      "equation": "", "discriminant": 0.0, "result": ""})
    assert out["discriminant"] == expected_d, (a, b, c, out["discriminant"])
    print(f"{a:>2} {b:>3} {c:>3}  d = {out['discriminant']:>4}  {out['result']}")

# 4**2 - 4*2*4 = 16 - 32 = -16, not -6. Both are negative, so the branch and the
# answer would look identical either way - test a boundary, not one case.
assert 4 ** 2 - 4 * 2 * 4 == -16
print("all three signs of d routed correctly")

## Part B — How many of the five nodes ran?

In [ ]:
# Five nodes are defined. How many run?
from langgraph.checkpoint.memory import InMemorySaver

app = graph.compile(checkpointer=InMemorySaver())
for a, b, c in ((4, -5, -4), (1, -4, 4), (2, 4, 4)):
    config = {"configurable": {"thread_id": f"{a},{b},{c}"}}
    app.invoke({"a": a, "b": b, "c": c,
                "equation": "", "discriminant": 0.0, "result": ""}, config=config)
    history = list(app.get_state_history(config))
    # snapshot.next names the node each step was about to run
    ran = [n for snap in reversed(history) for n in snap.next if n != "__start__"]
    print(f"a={a} b={b} c={c} -> {ran}")

# a=4 b=-5 c=-4 -> ['show_equation', 'calculate_discriminant', 'real_roots']
# a=1 b=-4 c=4  -> ['show_equation', 'calculate_discriminant', 'repeated_roots']
# a=2 b=4 c=4   -> ['show_equation', 'calculate_discriminant', 'no_real_roots']
#
# Three of five, every time, and the third one changes with the input. A
# parallel version of the same graph would run all five on every call.

In [ ]:
config = {"configurable": {"thread_id": "count-check"}}
app.invoke({"a": 4, "b": -5, "c": -4,
            "equation": "", "discriminant": 0.0, "result": ""}, config=config)
history = list(app.get_state_history(config))
ran = [n for snap in reversed(history) for n in snap.next if n != "__start__"]
assert ran == ["show_equation", "calculate_discriminant", "real_roots"]
assert "repeated_roots" not in ran and "no_real_roots" not in ran
print(f"{len(ran)} of 5 nodes ran; the two untaken branches cost nothing")

## Part C — What a routing function may return

Four experiments. The second one is the result worth remembering.

In [ ]:
# What is a routing function allowed to return? Four experiments on one graph.
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict):
    log: list

def build(router):
    graph = StateGraph(S)
    graph.add_node("start", lambda s: {})
    for name in ("a", "b"):
        graph.add_node(name, (lambda nm: lambda s: {"log": s["log"] + [nm]})(name))
        graph.add_edge(name, END)
    graph.add_edge(START, "start")
    graph.add_conditional_edges("start", router)
    return graph.compile()

print("a name that exists :", build(lambda s: "a").invoke({"log": []}))
print("a name with a typo :", build(lambda s: "typo").invoke({"log": []}))
print("END               :", build(lambda s: END).invoke({"log": []}))

def typed_router(state) -> Literal["a", "b"]:
    return "typo"

try:
    build(typed_router).invoke({"log": []})
except KeyError as e:
    print("typo, but annotated :", "KeyError", e)

# a name that exists : {'log': ['a']}
# a name with a typo : {'log': []}
# END                : {'log': []}
# typo, but annotated : KeyError 'typo'
#
# An unannotated router with a typo raises nothing and returns exactly what a
# router that chose to finish returns. It does log one line to stderr - "wrote
# to unknown channel branch:to:typo, ignoring it" - which is easy to miss in a
# notebook. Annotate the return type with `Literal` instead:
# LangGraph turns it into a path map and the typo becomes a KeyError. The
# annotation is the check, not decoration.

In [ ]:
# the typo and the deliberate END give the SAME result
typo = build(lambda s: "typo").invoke({"log": []})
ended = build(lambda s: END).invoke({"log": []})
assert typo == ended == {"log": []}
print("an un-annotated typo is indistinguishable from finishing early")

# with the annotation it raises instead
try:
    build(typed_router).invoke({"log": []})
    raise AssertionError("expected KeyError")
except KeyError as e:
    print("annotated ->", type(e).__name__, e)

# an explicit path map does the same job
try:
    graph = StateGraph(S)
    graph.add_node("start", lambda s: {})
    graph.add_node("a", lambda s: {"log": ["a"]})
    graph.add_edge(START, "start")
    graph.add_conditional_edges("start", lambda s: "typo", {"x": "a"})
    graph.add_edge("a", END)
    graph.compile().invoke({"log": []})
    raise AssertionError("expected KeyError")
except KeyError as e:
    print("path map ->", type(e).__name__, e)

## Part D — A router may choose a fan-out

In [ ]:
# "Exactly one branch" is the pattern, not a rule of the API: a router that
# returns a LIST fans out, and every name in the list runs.
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class Merged(TypedDict):
    log: Annotated[list, operator.add]       # two branches write it, so: reducer

graph = StateGraph(Merged)
graph.add_node("start", lambda s: {})
for name in ("a", "b"):
    graph.add_node(name, (lambda nm: lambda s: {"log": [nm]})(name))
    graph.add_edge(name, END)
graph.add_edge(START, "start")
graph.add_conditional_edges("start", lambda s: ["a", "b"])

print(graph.compile().invoke({"log": []}))

# {'log': ['a', 'b']}
#
# Both ran, in one super-step, exactly like lesson 007's fan-out. A conditional
# edge is better described as "the router decides which branches run" than as
# "the router picks one".

In [ ]:
out = graph.compile().invoke({"log": []})
assert sorted(out["log"]) == ["a", "b"]
print('"exactly one branch" is the pattern, not a rule of the API')

## Part E — Support routing, with branches of different lengths

In [ ]:
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

class ReviewState(TypedDict):
    review: str
    sentiment: str
    diagnosis: dict
    response: str

# A scripted classifier so the lesson runs with no key. With a real model this
# is one `with_structured_output(SentimentSchema)` call; see the next block.
SENTIMENT = {"The app is fast and the new dark mode is lovely.": "positive",
             "Third crash this week while uploading. Fix it.": "negative"}

def find_sentiment(state: ReviewState) -> dict:
    return {"sentiment": SENTIMENT.get(state["review"], "negative")}

def check_sentiment(state: ReviewState) -> Literal["positive_response",
                                                   "run_diagnosis"]:
    return "positive_response" if state["sentiment"] == "positive" else "run_diagnosis"

def positive_response(state: ReviewState) -> dict:
    return {"response": "Thank you for the kind words."}

def run_diagnosis(state: ReviewState) -> dict:
    return {"diagnosis": {"issue_type": "Bug", "tone": "angry", "urgency": "high"}}

def negative_response(state: ReviewState) -> dict:
    d = state["diagnosis"]
    return {"response": (f"We are sorry. A {d['urgency']}-urgency {d['issue_type']} "
                         f"report is open and an engineer will write today.")}

graph = StateGraph(ReviewState)
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("positive_response", END)
graph.add_edge("run_diagnosis", "negative_response")     # the longer branch
graph.add_edge("negative_response", END)

app = graph.compile()
for review in SENTIMENT:
    out = app.invoke({"review": review, "sentiment": "", "diagnosis": {},
                      "response": ""})
    print(out["sentiment"], "|", out["diagnosis"] or "-", "|", out["response"])

# positive | - | Thank you for the kind words.
# negative | {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'} | We are
# sorry. A high-urgency Bug report is open and an engineer will write today.
#
# The two branches are different lengths: positive answers in one node, negative
# diagnoses first and then answers. Nothing forces branches to match.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

app = graph.compile(checkpointer=InMemorySaver())
steps = {}
for i, review in enumerate(SENTIMENT):
    config = {"configurable": {"thread_id": f"r{i}"}}
    out = app.invoke({"review": review, "sentiment": "", "diagnosis": {},
                      "response": ""}, config=config)
    steps[out["sentiment"]] = max(s.metadata["step"]
                                  for s in app.get_state_history(config))
print(steps)
assert steps["positive"] == 2 and steps["negative"] == 3
assert steps["negative"] - steps["positive"] == 1
print("the negative branch is one super-step longer - it diagnoses first")

In [ ]:
# the drawing distinguishes the two kinds of edge
mermaid = app.get_graph().draw_mermaid()
solid = [l for l in mermaid.splitlines() if "-->" in l and "-.->" not in l]
dotted = [l.strip() for l in mermaid.splitlines() if "-.->" in l]
print("\n".join(dotted))
assert len(dotted) == 2        # the two branches the router may choose
print(f"{len(solid)} solid arrows, {len(dotted)} dotted")

## What this notebook established

- `add_conditional_edges(node, router)` replaces the single edge that would have
  left that node; the router reads the state and names the next node.
- Only the chosen branch runs: **3 of 5** nodes on every call, with only the
  third name changing.
- An un-annotated router with a typo in it **raises nothing** and returns the
  same thing as finishing early. Annotate it `-> Literal[...]`, or pass a path
  map, and the typo becomes a `KeyError`.
- A router may return a **list**, and then every name in it runs.
- Branches need not match: 2 super-steps for a positive review, 3 for a negative
  one. Conditional edges are drawn dotted.

## Exercises

1. Add a fourth branch for `a == 0` (not a quadratic at all) and extend the
   router's `Literal` to match.
2. Remove the `Literal` annotation from `check_condition`, misspell one branch
   name, and find out how long it takes you to notice.
3. Give the support workflow a third sentiment, `neutral`, with its own branch.
4. Make the router return `["positive_response", "run_diagnosis"]` and work out
   which key now needs a reducer.